# NB_PRF_01 · Perfilar las tablas de origen en bronze

**Paso 2 del framework. Solo lee bronze**: escribe únicamente en el esquema `perfil` y en `Files/perfil/`.

Por cada tabla de bronze (la carga exploratoria del paso 1) calcula, en una pasada por tabla:

| Qué | Para qué decisión |
|---|---|
| Filas, nulos, vacíos, espacios sobrantes | Reglas `IS NOT NULL`, `trim` en `TRANSFORM_EXPR` |
| Distintos, unicidad, claves candidatas simples y compuestas | `PRIMARY_KEY` y `SILVER_MODE` |
| Patrones de texto (enteros, decimales con coma, fechas `dd/MM/yyyy`, booleanos) | `TARGET_TYPE` y `TRANSFORM_EXPR` |
| Mínimo, máximo, percentiles, negativos, fechas futuras | Reglas `DROP` y `WARN` de `meta.DQ_RULE` |
| Dominios chicos y variantes de mayúsculas | Reglas `IN (...)` y normalización |
| PII por nombre y por contenido (email, CUIT/CUIL, CBU) | `IS_PII` y qué queda fuera de gold |
| Columnas candidatas a watermark y a baja lógica | `LOAD_TYPE`, `WATERMARK_COLUMN`, `SOURCE_FILTER` |

Si le pasás el catálogo del origen (`sql/01_catalogo_origen.sql`), suma tipos, PK, FK y descripciones declaradas.

Deja en el lakehouse por defecto:

| Dónde | Qué |
|---|---|
| `perfil.corrida`, `perfil.tabla`, `perfil.columna`, `perfil.clave_candidata` | El perfil en Delta, con historia por `corrida_id` (sirve para detectar cambios en el origen) |
| `Files/perfil/<origen>/<corrida_id>/perfil.json` | Lo mismo, compacto. Lo completa `NB_PRF_02` y es lo que lee Claude |

Los valores de ejemplo, mínimos y máximos de columnas que parecen PII **no se guardan**.

In [ ]:
# Parámetros
ORIGEN = "erp"                    # código del sistema de origen, como en meta.SOURCE_SYSTEM
TABLAS = "bronze.erp_*"           # tablas separadas por coma; admite * en el nombre
SOLO_ULTIMA_CORRIDA = True        # bronze acumula copias: perfila solo el último _run_id de cada tabla
MAX_FILAS_PERFIL = 2000000        # arriba de esto las distribuciones salen de una muestra (filas y claves no)
CATALOGO = ""                     # salida de sql/01_catalogo_origen.sql (carpeta parquet o archivo csv); "" = sin catálogo
PREFIJO_BRONZE = "erp_"           # convención ENTITY_CODE = <sistema>_<tabla>, para cruzar con el catálogo
MAPEO_TABLAS = "{}"               # JSON {"bronze.erp_cli": "dbo.Clientes"} para las que no siguen la convención
ESQUEMA_PERFIL = "perfil"
CARPETA_SALIDA = "/lakehouse/default/Files/perfil"
UMBRAL_TIPO = 0.995               # proporción de valores que tienen que cumplir un patrón para sugerir el tipo
UMBRAL_PATRON = 0.8               # proporción para marcar PII por contenido
MAX_DOMINIO = 20                  # hasta cuántos valores distintos se guarda el dominio completo
TOP_N = 5                         # valores más frecuentes por columna

# Igual que en NB_DOC_01 (Documentar_Modelo_Semantico): coincide por substring, sin distinguir mayúsculas
PATRONES_SENSIBLES = [
    "email", "mail", "dni", "cuit", "cuil", "documento", "telefono", "phone", "celular",
    "direccion", "address", "domicilio", "nombre", "apellido", "name", "cbu", "iban",
    "tarjeta", "card", "password", "pass", "token", "nacimiento", "birth", "salario", "sueldo",
]

In [ ]:
import fnmatch, itertools, json, os, re, unicodedata
from datetime import datetime, timezone
from pyspark.sql import functions as F

AUDITORIA = {"_run_id", "_ingested_at", "_source_file", "_file_modified_at"}
CORRIDA_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
INICIO = datetime.now(timezone.utc)
mapeo = {k.lower(): v for k, v in json.loads(MAPEO_TABLAS or "{}").items()}

# Patrones sobre texto (sintaxis Java). Se cuentan sobre los valores no nulos y no vacíos.
REGEX = {
    "entero":        r"^-?[0-9]+$",
    "decimal_punto": r"^-?[0-9]+\.[0-9]+$",
    "decimal_coma":  r"^-?[0-9]{1,3}(\.[0-9]{3})*,[0-9]+$|^-?[0-9]+,[0-9]+$",
    "fecha_iso":     r"^[0-9]{4}-[0-9]{2}-[0-9]{2}$",
    "fecha_barra":   r"^[0-9]{1,2}/[0-9]{1,2}/[0-9]{4}$",
    "timestamp_iso": r"^[0-9]{4}-[0-9]{2}-[0-9]{2}[ T][0-9]{2}:[0-9]{2}(:[0-9]{2}(\.[0-9]+)?)?$",
    "booleano":      r"(?i)^(true|false|si|sí|no|s|n|y|yes)$",
    "email":         r"^[^@\s]+@[^@\s]+\.[^@\s]+$",
    "cuit_cuil":     r"^(20|23|24|27|30|33|34)-?[0-9]{8}-?[0-9]$",
    "cbu":           r"^[0-9]{22}$",
}
WATERMARK_NOMBRE = re.compile(r"modif|actualiz|update|ultim|last|cambio|timestamp|rowversion", re.I)
BAJA_NOMBRE = re.compile(r"anulad|borrad|elimin|delet|activ|vigent|baja", re.I)
CLAVE_NOMBRE = re.compile(r"(^id|id$|_id_|^cod|codigo|nro|numero|key$)", re.I)
LINEA_NOMBRE = re.compile(r"linea|renglon|item|posicion|secuencia", re.I)
NUMERICOS = ("tinyint", "smallint", "int", "bigint", "float", "double")


def snake(nombre):
    """ClienteID -> cliente_id ; Razon_Social -> razon_social ; IDCliente -> id_cliente"""
    s = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    s = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", s)
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)
    return re.sub(r"_+", "_", re.sub(r"[^0-9A-Za-z]+", "_", s)).strip("_").lower()


def nombre_valido(c):
    """El mismo saneo que NB_MDP_10_bronze: así el catálogo cruza con las columnas de bronze."""
    return re.sub(r"[ ,;{}()\n\t=]+", "_", c.strip())


def es_sensible(nombre):
    n = nombre.lower()
    return next((p for p in PATRONES_SENSIBLES if p in n), None)


def tablas_a_perfilar():
    salida = []
    for patron in [t.strip() for t in TABLAS.split(",") if t.strip()]:
        esquema, nombre = patron.rsplit(".", 1)
        if "*" in nombre or "?" in nombre:
            for fila in spark.sql(f"SHOW TABLES IN {esquema}").collect():
                if not fila["isTemporary"] and fnmatch.fnmatch(fila["tableName"].lower(), nombre.lower()):
                    salida.append(f"{esquema}.{fila['tableName']}")
        else:
            salida.append(patron)
    return sorted(set(salida))


def leer_bronze(tabla):
    """Devuelve (df sin columnas de auditoría, _run_id perfilado o None)."""
    df = spark.table(tabla)
    corrida = None
    if SOLO_ULTIMA_CORRIDA and "_run_id" in df.columns and "_ingested_at" in df.columns:
        fila = (df.groupBy("_run_id").agg(F.max("_ingested_at").alias("m"))
                  .orderBy(F.desc("m")).first())
        if fila:
            corrida = fila["_run_id"]
            df = df.where(F.col("_run_id") == corrida)
    return df.drop(*[c for c in df.columns if c in AUDITORIA]), corrida


def tipo_simple(df, c):
    return df.schema[c].dataType.simpleString()

## 1. Catálogo del origen (opcional)

In [ ]:
catalogo = {}   # nombre normalizado de tabla -> {"objeto": esquema.tabla, "columnas": {col_bronze_lower: fila}}
if CATALOGO:
    lector = spark.read.option("header", "true") if CATALOGO.lower().endswith(".csv") else spark.read
    cat = (lector.csv(CATALOGO) if CATALOGO.lower().endswith(".csv") else lector.parquet(CATALOGO))
    cat = cat.toDF(*[c.lower() for c in cat.columns])
    for f in cat.collect():
        d = f.asDict()
        objeto = f"{d['esquema']}.{d['tabla']}"
        t = catalogo.setdefault(d["tabla"].lower(), {"objeto": objeto, "columnas": {}})
        t["columnas"][nombre_valido(d["columna"]).lower()] = d
    print(f"Catálogo: {len(catalogo)} tablas")


def buscar_catalogo(tabla_bronze):
    if tabla_bronze.lower() in mapeo:
        return catalogo.get(mapeo[tabla_bronze.lower()].split(".")[-1].lower())
    corto = tabla_bronze.split(".")[-1].lower()
    if PREFIJO_BRONZE and corto.startswith(PREFIJO_BRONZE.lower()):
        corto = corto[len(PREFIJO_BRONZE):]
    for clave, t in catalogo.items():
        if corto in (clave, clave.replace("_", ""), snake(clave)):
            return t
    return None


def es_verdadero(v):
    return str(v).strip().lower() in ("1", "true", "si", "sí", "yes")

## 2. Funciones de perfil

In [ ]:
def agregados(df, columnas):
    """Una sola pasada por tabla: todas las métricas de todas las columnas."""
    exprs = []
    for i, c in enumerate(columnas):
        col, t, p = F.col(f"`{c}`"), tipo_simple(df, c), f"c{i}__"
        exprs += [F.sum(col.isNull().cast("int")).alias(p + "nulos"),
                  F.approx_count_distinct(col).alias(p + "distintos")]
        if t == "string":
            lleno = col.isNotNull() & (F.trim(col) != "")
            exprs += [F.sum((F.trim(col) == "").cast("int")).alias(p + "vacios"),
                      F.sum((col != F.trim(col)).cast("int")).alias(p + "con_espacios"),
                      F.approx_count_distinct(F.upper(F.trim(col))).alias(p + "distintos_normalizados"),
                      F.min(F.length(col)).alias(p + "long_min"), F.max(F.length(col)).alias(p + "long_max"),
                      F.round(F.avg(F.length(col)), 1).alias(p + "long_prom"),
                      F.min(col).alias(p + "min"), F.max(col).alias(p + "max"),
                      F.sum((F.trim(col).rlike(r"^0[0-9]+$")).cast("int")).alias(p + "ceros_izq"),
                      F.max(F.length(F.regexp_extract(F.trim(col), r"[.,]([0-9]+)$", 1))).alias(p + "escala_max"),
                      F.max(F.length(F.regexp_replace(F.regexp_extract(F.trim(col), r"^-?([0-9.]+)", 1), r"\.", "")))
                       .alias(p + "enteros_max"),
                      # Fechas con barra: si la primera parte pasa de 12 es dd/MM; si la segunda, MM/dd
                      F.sum((F.split(F.trim(col), "/").getItem(0).cast("int") > 12).cast("int")).alias(p + "barra_p1_mayor12"),
                      F.sum((F.split(F.trim(col), "/").getItem(1).cast("int") > 12).cast("int")).alias(p + "barra_p2_mayor12"),
                      F.sum((lleno & F.trim(col).rlike(r"^(20|23|24|27)")).cast("int")).alias(p + "cuit_persona")]
            exprs += [F.sum((lleno & F.trim(col).rlike(rx)).cast("int")).alias(p + "pat_" + k) for k, rx in REGEX.items()]
        elif t in NUMERICOS or t.startswith("decimal"):
            exprs += [F.min(col).alias(p + "min"), F.max(col).alias(p + "max"),
                      F.sum((col < 0).cast("int")).alias(p + "negativos"),
                      F.sum((col == 0).cast("int")).alias(p + "ceros"),
                      F.percentile_approx(col.cast("double"), [0.01, 0.99], 1000).alias(p + "p01_p99")]
        elif t in ("date", "timestamp") or t.startswith("timestamp"):
            exprs += [F.min(col).alias(p + "min"), F.max(col).alias(p + "max"),
                      F.sum((col > F.current_timestamp()).cast("int")).alias(p + "futuras"),
                      F.sum((col < F.lit("1900-01-01").cast("date")).cast("int")).alias(p + "antiguas")]
            if t != "date":
                exprs.append(F.sum((F.date_format(col, "HH:mm:ss.SSS") != "00:00:00.000").cast("int"))
                              .alias(p + "con_hora"))
        elif t == "boolean":
            exprs.append(F.sum(col.cast("int")).alias(p + "verdaderos"))
    return df.agg(*exprs).first().asDict()


def sugerir_tipo(c, t, m, base):
    """Devuelve (tipo Spark para silver, TRANSFORM_EXPR o None, notas)."""
    ref, notas = f"`{c}`", []
    trim = f"trim({ref})" if m.get("con_espacios") else None
    if t in ("tinyint", "smallint", "int"):
        return "int", None, notas
    if t == "bigint" or t.startswith("decimal"):
        return t, None, notas
    if t in ("float", "double"):
        notas.append("double en el origen: si es un monto, decimal; confirmá la escala [dato a confirmar]")
        return "decimal(18,4)", None, notas
    if t.startswith("timestamp"):
        if base and not m.get("con_hora"):
            notas.append("todas las horas son 00:00: es una fecha")
            return "date", None, notas
        return "timestamp", None, notas
    if t in ("date", "boolean"):
        return t, None, notas
    if t != "string":
        return t, None, notas
    if not base:
        notas.append("columna vacía en la muestra")
        return "string", trim, notas

    pct = {k: (m.get("pat_" + k) or 0) / base for k in REGEX}
    if pct["entero"] >= UMBRAL_TIPO:
        if m.get("ceros_izq"):
            notas.append(f"{m['ceros_izq']} valores con ceros a la izquierda: es un código, queda string")
            return "string", trim, notas
        return ("int" if (m.get("enteros_max") or 0) <= 9 else "bigint"), trim, notas
    escala = max(int(m.get("escala_max") or 0), 2)
    precision = min(38, max(18, int(m.get("enteros_max") or 0) + escala))
    if pct["decimal_punto"] + pct["entero"] >= UMBRAL_TIPO and pct["decimal_punto"] > 0:
        return f"decimal({precision},{escala})", trim, notas
    if pct["decimal_coma"] + pct["entero"] >= UMBRAL_TIPO and pct["decimal_coma"] > 0:
        notas.append("decimales con coma y miles con punto")
        return (f"decimal({precision},{escala})",
                f"replace(replace(trim({ref}), '.', ''), ',', '.')", notas)
    if pct["fecha_iso"] >= UMBRAL_TIPO:
        return "date", f"to_date(trim({ref}), 'yyyy-MM-dd')", notas
    if pct["timestamp_iso"] >= UMBRAL_TIPO:
        return "timestamp", f"to_timestamp(trim({ref}))", notas
    if pct["fecha_barra"] >= UMBRAL_TIPO:
        if m.get("barra_p2_mayor12") and not m.get("barra_p1_mayor12"):
            return "date", f"to_date(trim({ref}), 'M/d/yyyy')", notas
        if not m.get("barra_p1_mayor12"):
            notas.append("fecha con barra ambigua (ningún día > 12): asumí dd/MM [dato a confirmar]")
        return "date", f"to_date(trim({ref}), 'd/M/yyyy')", notas
    if pct["booleano"] >= UMBRAL_TIPO:
        return ("boolean",
                f"CASE WHEN upper(trim({ref})) IN ('TRUE','SI','SÍ','S','Y','YES') THEN true "
                f"WHEN upper(trim({ref})) IN ('FALSE','NO','N') THEN false END", notas)
    return "string", trim, notas


def detectar_pii(c, t, m, base):
    """Devuelve (es_pii, motivo). Por nombre o por contenido."""
    if t == "string" and base:
        if (m.get("pat_email") or 0) / base >= UMBRAL_PATRON:
            return True, f"contenido: {100 * m['pat_email'] // base}% emails"
        if (m.get("pat_cuit_cuil") or 0) / base >= UMBRAL_PATRON:
            if m.get("cuit_persona"):
                return True, f"contenido: CUIT/CUIL, {100 * m['cuit_persona'] // base}% de personas (20/23/24/27)"
            return False, "contenido: CUIT de empresas solamente (30/33/34): no es dato personal"
        if (m.get("pat_cbu") or 0) / base >= UMBRAL_PATRON:
            return True, "contenido: CBU (22 dígitos)"
    patron = es_sensible(c)
    if patron:
        return True, f"nombre: contiene '{patron}' (revisar)"
    return False, None


def top_valores(df, c, n):
    filas = (df.where(F.col(f"`{c}`").isNotNull()).groupBy(f"`{c}`").count()
               .orderBy(F.desc("count")).limit(n).collect())
    return [{"valor": str(f[0]), "filas": int(f[1])} for f in filas]


def buscar_claves(df, candidatas, filas):
    """Claves simples exactas y, si no hay, pares y ternas. Todo sobre la tabla completa."""
    claves = []
    if not filas:
        return claves
    if candidatas:
        exactos = df.agg(*[F.countDistinct(f"`{c}`").alias(f"k{i}") for i, c in enumerate(candidatas)]).first()
        claves = [{"columnas": [c], "tipo": "simple", "distintos": int(exactos[f"k{i}"])}
                  for i, c in enumerate(candidatas) if exactos[f"k{i}"] == filas]
    return claves


def puntaje(f):
    """Cuánto parece una columna de clave: 0 es lo mejor. Ordena las claves candidatas."""
    t = f["tipo_sugerido"]
    if f["pii"]:
        return 9
    if LINEA_NOMBRE.search(f["columna"]):
        return 0
    entero = t in ("int", "bigint")
    if CLAVE_NOMBRE.search(f["columna"]):
        return 1 if entero else 2
    if entero:
        return 3
    if t == "string" and (f["long_max"] or 99) <= 20:
        return 4
    return 6 if t in ("date", "timestamp") else 8


def buscar_compuestas(df, columnas, filas, max_cols=6):
    for n in (2, 3):
        combos = list(itertools.combinations(columnas[:max_cols if n == 2 else 4], n))
        if not combos:
            continue
        r = df.agg(*[F.countDistinct(*[f"`{c}`" for c in cmb]).alias(f"k{i}") for i, cmb in enumerate(combos)]).first()
        encontradas = [{"columnas": list(cmb), "tipo": "compuesta", "distintos": int(r[f"k{i}"])}
                       for i, cmb in enumerate(combos) if r[f"k{i}"] == filas]
        if encontradas:
            return encontradas
    return []

## 3. Perfilar cada tabla

In [ ]:
filas_tabla, filas_columna, filas_clave, perfil_json = [], [], [], []

for tabla in tablas_a_perfilar():
    df_full, run_bronze = leer_bronze(tabla)
    df_full = df_full.persist()
    filas = df_full.count()
    muestreado = filas > MAX_FILAS_PERFIL
    df = df_full.sample(fraction=MAX_FILAS_PERFIL / filas, seed=42) if muestreado else df_full
    filas_muestra = df.count() if muestreado else filas
    columnas = list(df.columns)
    cat = buscar_catalogo(tabla)
    m_all = agregados(df, columnas) if columnas else {}

    cols_perfil = []
    for i, c in enumerate(columnas):
        t = tipo_simple(df, c)
        m = {k.split("__", 1)[1]: v for k, v in m_all.items() if k.startswith(f"c{i}__")}
        nulos, vacios = int(m.get("nulos") or 0), int(m.get("vacios") or 0)
        base = filas_muestra - nulos - vacios
        tipo_sug, transform, notas = sugerir_tipo(c, t, m, base)
        pii, pii_motivo = detectar_pii(c, t, m, base)
        cat_col = (cat or {}).get("columnas", {}).get(c.lower()) if cat else None

        if m.get("distintos_normalizados") and m["distintos_normalizados"] < (m.get("distintos") or 0):
            notas.append(f"{m['distintos']} valores distintos pero {m['distintos_normalizados']} "
                         "sin contar mayúsculas y espacios: normalizar con upper(trim())")
        if t == "string" and m.get("con_espacios"):
            notas.append(f"{m['con_espacios']} valores con espacios al principio o al final")

        dominio = None
        # Dominio: pocos valores que se repiten (un estado, una moneda), no una columna casi única
        if not pii and (m.get("distintos") or 0) <= min(MAX_DOMINIO, filas_muestra / 2):
            dominio = top_valores(df, c, MAX_DOMINIO + 1)
            if len(dominio) > MAX_DOMINIO:
                dominio = None
        tops = None if pii else (dominio[:TOP_N] if dominio else top_valores(df, c, TOP_N))

        watermark = None
        if tipo_sug in ("timestamp", "date") and nulos == 0 and WATERMARK_NOMBRE.search(c):
            watermark = "DATETIME"
        baja = bool(BAJA_NOMBRE.search(c)) and (m.get("distintos") or 99) <= 5

        p01_p99 = m.get("p01_p99") or [None, None]
        fila = {
            "corrida_id": CORRIDA_ID, "origen": ORIGEN, "tabla": tabla, "columna": c, "ordinal": i + 1,
            "tipo_bronze": t,
            "tipo_origen": (cat_col or {}).get("tipo"),
            "admite_nulos_origen": None if not cat_col else es_verdadero(cat_col.get("admite_nulos")),
            "es_pk_declarada": bool(cat_col and es_verdadero(cat_col.get("es_pk"))),
            "fk_declarada": (f"{cat_col['fk_esquema_ref']}.{cat_col['fk_tabla_ref']}.{cat_col['fk_columna_ref']}"
                             if cat_col and cat_col.get("fk_tabla_ref") else None),
            "descripcion_origen": (cat_col or {}).get("descripcion_columna"),
            "filas": filas, "nulos": nulos, "pct_nulos": round(100 * nulos / filas_muestra, 2) if filas_muestra else None,
            "vacios": vacios, "con_espacios": m.get("con_espacios"),
            "distintos": m.get("distintos"),
            "pct_distintos": round(100 * (m.get("distintos") or 0) / filas_muestra, 2) if filas_muestra else None,
            "es_unica": False,
            "min": None if pii or m.get("min") is None else str(m["min"]),
            "max": None if pii or m.get("max") is None else str(m["max"]),
            "p01": None if pii or p01_p99[0] is None else float(p01_p99[0]),
            "p99": None if pii or p01_p99[1] is None else float(p01_p99[1]),
            "negativos": m.get("negativos"), "ceros": m.get("ceros"),
            "futuras": m.get("futuras"), "antiguas": m.get("antiguas"),
            "long_min": m.get("long_min"), "long_max": m.get("long_max"), "long_prom": m.get("long_prom"),
            "ceros_izq": m.get("ceros_izq"),
            "patrones": json.dumps({k: round((m.get("pat_" + k) or 0) / base, 4) for k in REGEX
                                    if base and m.get("pat_" + k)}) if t == "string" else None,
            "tipo_sugerido": tipo_sug, "transform_sugerida": transform, "columna_silver_sugerida": snake(c),
            "pii": pii, "pii_motivo": pii_motivo,
            "top_valores": json.dumps(tops, ensure_ascii=False) if tops is not None else None,
            "dominio": json.dumps(dominio, ensure_ascii=False) if dominio is not None else None,
            "candidata_watermark": watermark, "candidata_baja_logica": baja,
            "notas": json.dumps(notas, ensure_ascii=False) if notas else None,
            "muestreado": muestreado,
        }
        cols_perfil.append(fila)

    # Claves: candidatas simples = sin nulos y casi todos distintos (approx tiene ~5% de error)
    no_flotantes = [f for f in cols_perfil if f["tipo_bronze"] not in ("float", "double", "boolean")]
    candidatas = [f["columna"] for f in no_flotantes
                  if f["nulos"] == 0 and f["vacios"] == 0 and (f["distintos"] or 0) >= 0.9 * filas_muestra]
    claves = buscar_claves(df_full, candidatas, filas)
    for k in claves:
        next(f for f in cols_perfil if f["columna"] == k["columnas"][0])["es_unica"] = True
    if not claves and filas:
        orden = sorted([f for f in no_flotantes if f["nulos"] == 0 and f["vacios"] == 0],
                       key=lambda f: -(f["distintos"] or 0))
        claves = buscar_compuestas(df_full, [f["columna"] for f in orden], filas)
    pk_decl = [f["columna"] for f in cols_perfil if f["es_pk_declarada"]]
    if pk_decl:
        distintos = df_full.agg(F.countDistinct(*[f"`{c}`" for c in pk_decl]).alias("d")).first()["d"]
        claves.insert(0, {"columnas": pk_decl, "tipo": "declarada", "distintos": int(distintos)})
    # La preferida es la que usa NB_PRF_02 como padre de relaciones y la que Claude propone como PRIMARY_KEY
    por_nombre = {f["columna"]: f for f in cols_perfil}
    claves.sort(key=lambda k: (k["tipo"] != "declarada", sum(puntaje(por_nombre[c]) for c in k["columnas"]),
                               len(k["columnas"])))
    preferida = next((k for k in claves if k["distintos"] == filas), None)
    for k in claves:
        filas_clave.append({"corrida_id": CORRIDA_ID, "origen": ORIGEN, "tabla": tabla,
                            "columnas": ",".join(k["columnas"]), "tipo": k["tipo"], "filas": filas,
                            "distintos": k["distintos"], "es_unica": k["distintos"] == filas,
                            "preferida": k is preferida})

    descripcion_tabla = None
    if cat and cat["columnas"]:
        descripcion_tabla = next(iter(cat["columnas"].values())).get("descripcion_tabla")
    fila_t = {"corrida_id": CORRIDA_ID, "origen": ORIGEN, "tabla": tabla,
              "tabla_origen": cat["objeto"] if cat else None, "descripcion_origen": descripcion_tabla,
              "run_id_bronze": run_bronze, "filas": filas, "columnas": len(columnas), "muestreado": muestreado,
              "columnas_pii": sum(1 for f in cols_perfil if f["pii"])}
    filas_tabla.append(fila_t)
    filas_columna += cols_perfil

    claves_txt = f"{preferida['tipo']}: {','.join(preferida['columnas'])}" if preferida else "SIN CLAVE ÚNICA"
    print(f"{tabla}: {filas} filas, {len(columnas)} columnas, clave [{claves_txt}] de {len(claves)} candidatas, "
          f"PII {fila_t['columnas_pii']}{' (muestra)' if muestreado else ''}")
    df_full.unpersist()

## 4. Guardar en Delta y en `perfil.json`

In [ ]:
ESQUEMA_COLUMNA = """corrida_id string, origen string, tabla string, columna string, ordinal int,
    tipo_bronze string, tipo_origen string, admite_nulos_origen boolean, es_pk_declarada boolean,
    fk_declarada string, descripcion_origen string, filas bigint, nulos bigint, pct_nulos double,
    vacios bigint, con_espacios bigint, distintos bigint, pct_distintos double, es_unica boolean,
    min string, max string, p01 double, p99 double, negativos bigint, ceros bigint, futuras bigint,
    antiguas bigint, long_min int, long_max int, long_prom double, ceros_izq bigint, patrones string,
    tipo_sugerido string, transform_sugerida string, columna_silver_sugerida string, pii boolean,
    pii_motivo string, top_valores string, dominio string, candidata_watermark string,
    candidata_baja_logica boolean, notas string, muestreado boolean"""
ESQUEMA_TABLA = """corrida_id string, origen string, tabla string, tabla_origen string,
    descripcion_origen string, run_id_bronze string, filas bigint, columnas int, muestreado boolean,
    columnas_pii int"""
ESQUEMA_CLAVE = """corrida_id string, origen string, tabla string, columnas string, tipo string,
    filas bigint, distintos bigint, es_unica boolean, preferida boolean"""
ESQUEMA_CORRIDA = """corrida_id string, origen string, inicio timestamp, fin timestamp, tablas int,
    parametros string"""


def a_delta(filas, esquema, tabla):
    campos = [c.strip().split(" ")[0] for c in esquema.replace("\n", " ").split(",")]
    datos = [tuple(f.get(k) for k in campos) for f in filas]
    (spark.createDataFrame(datos, esquema).write.format("delta").mode("append")
          .option("mergeSchema", "true").saveAsTable(f"{ESQUEMA_PERFIL}.{tabla}"))


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ESQUEMA_PERFIL}")
parametros = {"tablas": TABLAS, "solo_ultima_corrida": SOLO_ULTIMA_CORRIDA, "max_filas_perfil": MAX_FILAS_PERFIL,
              "catalogo": CATALOGO, "umbral_tipo": UMBRAL_TIPO, "max_dominio": MAX_DOMINIO}
a_delta(filas_columna, ESQUEMA_COLUMNA, "columna")
a_delta(filas_tabla, ESQUEMA_TABLA, "tabla")
a_delta(filas_clave, ESQUEMA_CLAVE, "clave_candidata")
a_delta([{"corrida_id": CORRIDA_ID, "origen": ORIGEN, "inicio": INICIO, "fin": datetime.now(timezone.utc),
          "tablas": len(filas_tabla), "parametros": json.dumps(parametros)}], ESQUEMA_CORRIDA, "corrida")

# perfil.json: lo que lee Claude. Sin campos vacíos, para que sea corto.
def limpio(d, sacar=("corrida_id", "origen", "tabla")):
    salida = {}
    for k, v in d.items():
        if k in sacar or v is None or v is False:
            continue
        if k == "top_valores" and d.get("dominio"):
            continue   # el dominio ya tiene todos los valores
        if k in ("patrones", "top_valores", "dominio", "notas"):
            v = json.loads(v)
        salida[k] = v
    return salida


perfil = {"origen": ORIGEN, "corrida_id": CORRIDA_ID,
          "generado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
          "parametros": parametros, "tablas": [], "relaciones": []}
for t in filas_tabla:
    perfil["tablas"].append({
        **limpio(t, sacar=("corrida_id", "origen")),
        # Las 3 mejores alcanzan: en tablas chicas casi todas las columnas son únicas
        "claves_candidatas": [{"columnas": k["columnas"].split(","), "tipo": k["tipo"], "es_unica": k["es_unica"],
                               "preferida": k["preferida"]}
                              for k in filas_clave if k["tabla"] == t["tabla"]][:3],
        "columnas": [limpio(c) for c in filas_columna if c["tabla"] == t["tabla"]],
    })

salida = os.path.join(CARPETA_SALIDA, re.sub(r"[^\w\-]+", "_", ORIGEN), CORRIDA_ID)
os.makedirs(salida, exist_ok=True)
with open(os.path.join(salida, "perfil.json"), "w", encoding="utf-8") as f:
    json.dump(perfil, f, ensure_ascii=False, indent=1, default=str)

print(f"Corrida {CORRIDA_ID}: {len(filas_tabla)} tablas, {len(filas_columna)} columnas")
print(f"Listo en {salida}. Seguí con NB_PRF_02_relaciones (corrida_id = {CORRIDA_ID}).")

In [ ]:
notebookutils.notebook.exit(json.dumps({"corrida_id": CORRIDA_ID, "tablas": len(filas_tabla),
                                        "carpeta": salida}, ensure_ascii=False))